In [6]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
model = ChatOpenAI()

In [12]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    score: str

In [13]:
def create_outline(state: BlogState) -> BlogState:
    # fetch title 
    title = state['title']

    # call llm gen outline
    prompt = f"Generate a detailed outline fro a blog on the topic -  {title}"
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

def create_blog(state: BlogState) -> BlogState:
    # fetch outline and title
    outline = state['outline']
    title = state['title']

    # call llm gen blog content
    prompt = f"Write a detailed blog post based on the title - {title}  using the follwing outline - \n{outline}"
    content = model.invoke(prompt).content

    # update state
    state['content'] = content

    return state

def evaluate_blog(state: BlogState) -> BlogState:
    # fetch outline and title
    outline = state['outline']

    # call llm gen blog content
    prompt = f"Based on this outline evaluate my blog and rate it give me score out of 10 - \n{outline}"
    content = model.invoke(prompt).content

    # update state
    state['score'] = content

    return state


In [16]:
# create graph
graph = StateGraph(BlogState)

# add nodes
graph.add_node('createOutline', create_outline)
graph.add_node('createBlog', create_blog)
graph.add_node('evaluateBlog', evaluate_blog)



# add edges
graph.add_edge(START, "createOutline")
graph.add_edge("createOutline", "createBlog")    
graph.add_edge("createBlog", "evaluateBlog")    
graph.add_edge("evaluateBlog", END)    

# compile the graph 
workflow  = graph.compile()

# invoke the graph with an initial state
state = {'title':'Rise of AI in India'}
state = workflow.invoke(state)
state


{'title': 'Rise of AI in India',
 'outline': "I. Introduction\n    A. Brief overview of artificial intelligence (AI)\n    B. Explanation of how AI is gaining popularity in India\n    C. Thesis statement: The rise of AI in India is transforming various industries and shaping the future of technology\n\nII. History of AI in India\n    A. Early developments in AI technology in India\n    B. Key milestones in the adoption of AI in India \n    C. Leading research institutions and organizations contributing to the growth of AI in India\n\nIII. Impact of AI on Indian Industries\n    A. Healthcare\n        1. Use of AI in medical diagnosis and treatment\n        2. AI-powered healthcare solutions improving patient care and outcomes\n    B. Education\n        1. AI in personalized learning and education technology\n        2. AI-driven tools to enhance student engagement and performance\n    C. Agriculture\n        1. Applications of AI in precision farming and crop management\n        2. AI so